# 🎙️ Семинар: Анализ и аугментации речевых данных

Изучаем базовые техники анализа и аугментации для обработки аудио в PyTorch.

## 📋 План:
1. 📊 Визуализация: waveform, спектр, мел-спектрограмма
2. 🔊 Гауссов шум
3. ⏱️ Time Stretching
4. 🎼 Pitch Shifting
5. 🔉 Volume
6. 🏠 Room Impulse Response
7. 🌫️ Фоновый шум (SNR)
8. 🎭 SpecAugment

In [ ]:
from matplotlib import pyplot as plt
from IPython import display

import torch
import torch.nn.functional as F
from torch import nn
from torch import distributions
import torchaudio

import numpy as np
import librosa

## Загрузка аудиофайла

В этой ячейке мы загружаем аудиофайл формата WAV с помощью `torchaudio.load()`.
Функция возвращает тензор с аудиоданными (`wav`) и частоту дискретизации (`sr`).

Параметры:
- `wav`: тензор формы `[channels, time_samples]`
- `sr`: sample rate (частота дискретизации), по умолчанию 22050 Гц

In [ ]:
wav, sr = torchaudio.load('LJ001-0001.wav')

In [ ]:
sr

In [ ]:
print(wav[:10])

In [ ]:
wav.dtype

In [ ]:
wav.shape

## Функция визуализации аудио

Функция `visualize_audio` отображает:
1. Временной сигнал (waveform) — зависимость амплитуды от времени
2. Аудиоплеер для прослушивания непосредственно в ноутбуке

Если аудио многоканальное, выполняется усреднение по каналам для получения моно-сигнала.

In [ ]:
def visualize_audio(wav: torch.Tensor, sr: int = 22050):
    # Average all channels
    if wav.dim() == 2:
        # Any to mono audio convertion
        wav = wav.mean(dim=0)

    plt.figure(figsize=(20, 5))
    plt.plot(wav, alpha=.7, c='green')
    plt.grid()
    plt.xlabel('Time', size=20)
    plt.ylabel('Amplitude', size=20)
    plt.show()

    display.display(display.Audio(wav, rate=sr))

In [ ]:
visualize_audio(wav)

In [ ]:
n_fft = 1024

In [ ]:
spectrum = torch.fft.rfft(wav, n=n_fft)

In [ ]:
spectrum.dtype

In [ ]:
spectrum

## Вычисление спектрограммы (мощности)

Спектрограмма — это квадрат модуля спектра: `|X(f)|²`.
Эквивалентно можно вычислить через `torch.view_as_real` и норму по последней оси.

Проверяем эквивалентность двух способов вычисления через `torch.allclose`.

In [ ]:
spectrogram = spectrum.abs().pow(2)

In [ ]:
spectrogram_v2 = torch.view_as_real(spectrum).norm(dim=-1).pow(2)

In [ ]:
assert torch.allclose(spectrogram, spectrogram_v2)

## Визуализация спектра мощности

График показывает распределение энергии сигнала по частотам.
По оси X — частота (Гц), по оси Y — квадрат амплитуды (мощность).

In [ ]:
plt.figure(figsize=(20, 5))
plt.plot(spectrogram.squeeze(), c='green')
plt.grid()
plt.xlabel('Frequency (Hz)', size=20)
plt.ylabel('Magnitude$^2$', size=20)
plt.show()

## Окно Ханна (Hann Window)

Для уменьшения спектральных утечек при анализе коротких сегментов сигнала применяем оконную функцию.
Окно Ханна плавно обнуляет сигнал на краях окна, снижая артефакты преобразования Фурье.

In [ ]:
window_size = n_fft
window = torch.hann_window(window_size)

plt.figure(figsize=(20, 5))
plt.plot(window, c='green')
plt.grid()
plt.show()

## Сравнение: сырой сигнал vs оконированный

На левом графике — исходный обрезанный сигнал, на правом — тот же сигнал, умноженный на окно Ханна.
Обратите внимание на плавное затухание амплитуды к краям у оконированного сигнала.

In [ ]:
clipped_wav = wav[:, :window_size]
windowed_clipped_wav = window * clipped_wav

fig, axes = plt.subplots(1, 2, figsize=(20, 5))

axes[0].plot(clipped_wav.squeeze(), c='green')
axes[0].set_title('Raw Audio', size=20)

axes[1].plot(windowed_clipped_wav.squeeze(), c='green')
axes[1].set_title('Windowed Audio', size=20)

for i in range(2):
    axes[i].grid()
    axes[i].set_xlabel('Time', size=20)
    axes[i].set_ylabel('Amplitude', size=20)

plt.show()

## Влияние окна на спектр

Сравнение спектрограмм до и после применения окна.
Оконная функция снижает боковые лепестки в частотной области, делая спектр более «чистым» и интерпретируемым.

In [ ]:
spectrogram = torch.fft.rfft(clipped_wav).abs().pow(2)
windowed_spectrogram = torch.fft.rfft(windowed_clipped_wav).abs().pow(2)

fig, axes = plt.subplots(1, 2, figsize=(20, 5))

axes[0].plot(spectrogram.squeeze(), c='green')
axes[0].set_title('Spectrogram of Raw Audio', size=20)

axes[1].plot(windowed_spectrogram.squeeze(), c='green')
axes[1].set_title('Spectrogram of Windowed Audio', size=20)

for i in range(2):
    axes[i].grid()
    axes[i].set_xlabel('Frequency (Hz)', size=20)

plt.show()

## Вычисление STFT

Short-Time Fourier Transform разбивает сигнал на перекрывающиеся окна и применяет FFT к каждому.
Параметры:
- `n_fft=1024`: размер окна
- `hop_length=256`: шаг между окнами (25% перекрытия)
- `center=False`: без дополнения нулями по краям
- `return_complex=False`: возвращает вещественное представление [real, imag]

Результат: тензор формы `[batch, channels, freq_bins, time_frames]`

In [ ]:
n_fft = 1024
window_size = n_fft
hop_size = 256
window = torch.hann_window(n_fft)

In [ ]:
2048 / 22050

In [ ]:
spectrum = torch.stft(
    wav,
    n_fft=n_fft,
    hop_length=hop_size,
    win_length=window_size,
    window=window,

    # We don't want to pad input signal
    center=False,

    # Take first (n_fft // 2 + 1) frequencies
    onesided=True,

    # Apply torch.view_as_real on each window
    return_complex=False,
)

In [ ]:
spectrum.shape

In [ ]:
spectrogram = spectrum.norm(dim=-1).pow(2)
spectrogram.shape # [количество файлов, частотный бины, количество фреймоы - временных окон)

## Визуализация спектрограммы (логарифмическая шкала)

Отображаем логарифм мощности спектрограммы для лучшего восприятия динамического диапазона.
- Ось X: время (кадры)
- Ось Y: частота (Гц)
- Цвет: логарифм мощности (ярче = больше энергии)

In [ ]:
plt.figure(figsize=(20, 5))
plt.imshow(spectrogram.squeeze().log(), origin='lower')
plt.xlabel('Time', size=20)
plt.ylabel('Frequency (Hz)', size=20)
plt.show()

## Преобразование в мел-шкалу

Человеческое восприятие частот нелинейно: мы лучше различаем низкие частоты.
Мел-шкала аппроксимирует эту особенность.

`torchaudio.transforms.MelScale` применяет линейную проекцию спектральных бинов в 80 мел-каналов.
Матрица `fb` (filterbank) показывает, как каждый мел-бин агрегирует частоты из линейной шкалы.

In [ ]:
mel_scaler = torchaudio.transforms.MelScale(
    n_mels=80,
    sample_rate=22_050,
    n_stft=n_fft // 2 + 1
)

In [ ]:
mel_scaler.fb.shape

In [ ]:
plt.figure(figsize=(20, 5))
plt.imshow(mel_scaler.fb.T)
plt.xlabel('Hertz Scale', size=20)
plt.ylabel('Mels Scale', size=20)
plt.gca().invert_yaxis()
plt.show()

In [ ]:
mel_spectrogram = mel_scaler(spectrogram)

In [ ]:
mel_spectrogram.shape

## Визуализация мел-спектрограммы

Мел-спектрограмма — компактное представление аудио, широко используемое в ASR, TTS и аудио-классификации.
Ось Y теперь отображает мел-частоты, что лучше соответствует слуховому восприятию.

In [ ]:
plt.figure(figsize=(20, 5))
plt.imshow(mel_spectrogram.squeeze().log(), origin='lower')
plt.xlabel('Time', size=20)
plt.ylabel('Mels', size=20)
plt.show()

## Единый интерфейс: MelSpectrogram

`torchaudio.transforms.MelSpectrogram` объединяет все предыдущие шаги (окно, STFT, мел-проекция) в один модуль.
Проверяем, что результат совпадает с ручным вычислением через `torch.allclose`.

In [ ]:
featurizer = torchaudio.transforms.MelSpectrogram(
    sample_rate=22_050,
    n_fft=1024,
    win_length=1024,
    hop_length=256,
    n_mels=80,
    window_fn=torch.hann_window,
    center=False
)

In [ ]:
assert torch.allclose(mel_spectrogram, featurizer(wav))

In [ ]:
def visualize_spectrogram(wav: torch.Tensor, sr: int = 22050):
    spectrum = torch.stft(
        wav,
        n_fft=1024,
        hop_length=256,
        win_length=1024,
        window=torch.hann_window(1024),

        # We don't want to pad input signal
        center=False,

        # Take first (n_fft // 2 + 1) frequencies
        onesided=True,

        # Apply torch.view_as_real on each window
        return_complex=False,
    )
    spectrogram = spectrum.norm(dim=-1).pow(2)

    plt.figure(figsize=(20, 5))
    plt.imshow(spectrogram.squeeze().log(), origin='lower')
    plt.xlabel('Time', size=20)
    plt.ylabel('Frequency (Hz)', size=20)
    plt.show()

## Добавление гауссова шума

Простая аугментация: добавляем шум ~ N(0, 0.05) к каждому отсчету.
Помогает модели стать устойчивее к фоновым помехам.

In [ ]:
noiser = distributions.Normal(0, 0.05)

In [ ]:
augumented_wav_1 = wav +  0.1 * noiser.sample(wav.size())

In [ ]:
visualize_audio(augumented_wav_1)

In [ ]:
visualize_spectrogram(augumented_wav_1)

## Временное растяжение (Time Stretch)

Изменяем длительность аудио без изменения высоты тона.
- `librosa.effects.time_stretch(..., rate=2.0)`: ускорение в 2 раза
- Простое прореживание `wav[:, ::2]` — грубый аналог, но с артефактами

In [ ]:
simple_stretch = wav[:, ::2]
visualize_audio(simple_stretch)

In [ ]:
visualize_spectrogram(simple_stretch)

In [ ]:
augumented_wav_2 = librosa.effects.time_stretch(wav.numpy().squeeze(), rate=2.0)
augumented_wav_2 = torch.from_numpy(augumented_wav_2)
visualize_audio(augumented_wav_2)

## Сдвиг высоты тона (Pitch Shift)

Изменяем высоту тона без изменения длительности.
`librosa.effects.pitch_shift(wav, sr, n_steps=-5)` понижает тон на 5 полутонов.
Полезно для увеличения разнообразия вокальных данных.

In [ ]:
augumented_wav_3 = librosa.effects.pitch_shift(wav.numpy().squeeze(), sr=sr, n_steps=2)
augumented_wav_3 = torch.from_numpy(augumented_wav_3)

In [ ]:
visualize_audio(augumented_wav_3)

## Изменение громкости

`torchaudio.transforms.Vol` масштабирует амплитуду сигнала.
`gain=0.2` уменьшает громкость в 5 раз.
Моделирует вариации уровня записи и расстояния до микрофона.

In [ ]:
voler = torchaudio.transforms.Vol(gain=3, gain_type='amplitude')
augumented_wav_4 = voler(wav)
visualize_audio(augumented_wav_4)

## Свёртка с импульсной характеристикой помещения (RIR)

Имитация акустики помещения через свёртку с записанной импульсной характеристикой (Room Impulse Response).
Функция `simulate`:
1. Дополняет сигнал нулями для full convolution
2. Переворачивает RIR (т.к. `conv1d` выполняет кросс-корреляцию)
3. Нормализует пиковую амплитуду

Результат: аудио с эффектом реверберации «спальня».

In [ ]:
rir, rir_sr = torchaudio.load('h001_Bedroom_65.wav')
visualize_audio(rir, rir_sr)

In [ ]:
def simulate(audio: torch.Tensor, rir: torch.Tensor):
    left_pad = right_pad = rir.shape[-1] - 1

    # Since torch.conv do cross-correlation (not convolution) we need to flip kernel
    flipped_rir = rir.squeeze().flip(0)

    audio = F.pad(audio, [left_pad, right_pad]).view(1, 1, -1)
    convolved_audio = torch.conv1d(audio, flipped_rir.view(1, 1, -1)) \
        .squeeze()

    # peak normalization
    if convolved_audio.abs().max() > 1:
        convolved_audio /= convolved_audio.abs().max()

    return convolved_audio

In [ ]:
augumented_wav_5 = simulate(wav, rir)
visualize_audio(augumented_wav_5)

In [ ]:
visualize_spectrogram(augumented_wav_5)

## Добавление фонового шума с контролем SNR

Более реалистичная аугментация: добавляем внешний шум (например, запись трубы) с заданным отношением сигнал/шум (SNR).

Алгоритм:
1. Вычисляем энергию сигнала и шума
2. Масштабируем шум коэффициентом `alpha`, соответствующим целевому SNR (в дБ)
3. Обрезаем/дополняем сигналы до одинаковой длины
4. Складываем и ограничиваем диапазон [-1, 1]

Формула: `alpha = (E_audio / E_noise) * 10^(-SNR_dB / 20)`

In [ ]:
filename = librosa.ex('trumpet')
y, sr = librosa.load(filename)

noise = y

visualize_audio(torch.from_numpy(noise))

In [ ]:
visualize_spectrogram(torch.from_numpy(noise))

In [ ]:
noize_level = torch.Tensor([10])  # [0, 40]

noize_energy = torch.norm(torch.from_numpy(noise))
audio_energy = torch.norm(wav)

alpha = (audio_energy / noize_energy) * torch.pow(10, -noize_level / 20)

clipped_wav = wav[..., :noise.shape[0]]

augumented_wav_6 = clipped_wav + alpha * torch.from_numpy(noise)

# In some cases the resulting sound may go beyond [-1, 1]
# So, clamp it :)
augumented_wav_6 = torch.clamp(augumented_wav_6, -1, 1)

visualize_audio(augumented_wav_6)

In [ ]:
visualize_spectrogram(augumented_wav_6)

## SpecAugment: маскирование спектрограммы

Поскольку мел-спектрограмма технически является **изображением** (время × частота × интенсивность), к ней можно применять идеи из компьютерного зрения.

### 📄 Оригинал: [SpecAugment (Park et al., 2019)](https://arxiv.org/pdf/1904.08779.pdf)

### 🔧 Типы маскирования:
| Тип | Реализация в torchaudio | Эффект |
|-----|-------------------------|--------|
| **Frequency Masking** | `torchaudio.transforms.FrequencyMasking(freq_mask_param)` | Горизонтальные полосы — "выключаем" диапазоны частот |
| **Time Masking** | `torchaudio.transforms.TimeMasking(time_mask_param)` | Вертикальные полосы — "вырезаем" временные отрезки |
| **Cutout** (опционально) | кастомная реализация | Случайные прямоугольные области |

### 💡 Зачем это нужно?
- Заставляет модель учиться на **частичной информации**
- Повышает устойчивость к пропускам в сигнале
- Работает как регуляризатор, аналогично Dropout

> ⚠️ SpecAug применяется **на уровне признаков** (спектрограммы), а не waveform. Убедитесь, что вы сначала вычислили `MelSpectrogram`.

> 🔄 Существуют разные реализации: количество масок, максимальная ширина, применение к логарифмической шкале — экспериментируйте!

In [ ]:
mel_spectrogramer = torchaudio.transforms.MelSpectrogram(
    sample_rate=22050,
    n_fft=1024,
    win_length=1024,
    hop_length=256,
    n_mels=80,
)

mel_spectrogram = mel_spectrogramer(wav)
log_mel_spectrogram = torch.log(mel_spectrogram).squeeze()
plt.figure(figsize=(20, 5))
plt.imshow(log_mel_spectrogram, origin='lower')
plt.show()

In [ ]:
specaug = nn.Sequential(
    torchaudio.transforms.FrequencyMasking(20),
    torchaudio.transforms.TimeMasking(10),
)

augumented_log_mel_spectrogram = specaug(log_mel_spectrogram)

plt.figure(figsize=(20, 5))
plt.imshow(augumented_log_mel_spectrogram, origin='lower')
plt.show()